In [1]:
!pip install sentence-transformers pandas scikit-learn

In [2]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def get_similarity(s1, s2):
    emb1 = model.encode(s1)
    emb2 = model.encode(s2)
    return cosine_similarity([emb1], [emb2])[0][0]

In [5]:
def word_overlap(t, u):
    t_words = set(t.lower().split())
    u_words = set(u.lower().split())

    if len(t_words) == 0:
        return 0

    return len(t_words & u_words) / len(t_words)

In [6]:
def length_diff(t, u):
    return abs(len(t.split()) - len(u.split()))

In [45]:
def compute_score(target, user, sim):
    if not user:
        return 0

    t = target.lower()
    u = user.lower()

    overlap = word_overlap(t, u)

    # Strong match
    if sim > 0.9:
        return 5

    # Good match
    elif sim > 0.75:
        return 4

    # Partial meaning
    elif sim > 0.6:
        return 3

    # Weak relation
    elif sim > 0.4:
        return 2

    # Very poor
    else:
        return 1

In [46]:
import pandas as pd
# Path to the AutoEIT transcription dataset (update this to your local path)
file_path = "path/to/AutoEIT_Sample_Transcriptions.xlsx"
xls = pd.ExcelFile(file_path)

In [47]:
xls.sheet_names

['Info', '38010-2A', '38011-1A', '38012-2A', '38015-1A']

In [48]:
df = pd.read_excel(xls, sheet_name=xls.sheet_names[1])
df.head()

,Sentence,Stimulus,Transcription,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,1,Quiero cortarme el pelo (7),NaN,NaN,NaN,NaN,038010_EIT-2A.mp3
1,2,El libro está en la mesa (7),NaN,NaN,NaN,NaN,NaN
2,3,El carro lo tiene Pedro (8),NaN,NaN,NaN,NaN,NaN
3,4,El se ducha cada mañana (9),NaN,NaN,NaN,NaN,NaN
4,5,¿Qué dice usted que va a hacer hoy? (9),NaN,NaN,NaN,NaN,NaN


In [49]:
def clean_target(text):
    if pd.isna(text):
        return ""
    return text.split("(")[0].strip()

df_targets = df.copy()
df_targets["target"] = df_targets["Stimulus"].apply(clean_target)

df_targets = df_targets[["Sentence", "target"]]

In [50]:
df_transcript = pd.read_csv("../results/final_transcriptions.csv")
df_transcript.head()

,file,sentence_id,start,end,transcript
0,audio_samples/038010_EIT-2A.mp3,1,81.58,82.44,Le llamaré mañana.
1,audio_samples/038010_EIT-2A.mp3,2,107.74,110.34,"A veces, toman su perro para un parque."
2,audio_samples/038010_EIT-2A.mp3,3,132.99,156.72,Vamos a jugar a la voleybal.
3,audio_samples/038010_EIT-2A.mp3,4,193.06,199.56,¿Qué dice usted que va a hacer hoy?
4,audio_samples/038010_EIT-2A.mp3,5,201.96,208.64,Dudo que sepa manejar muy bien


In [51]:
df_transcript = df_transcript.rename(columns={
    "sentence_id": "Sentence"
})

In [52]:
def find_best_match(target, df_transcript):
    best_score = -1
    best_transcript = ""

    for _, row in df_transcript.iterrows():
        transcript = row["transcript"]

        sim = get_similarity(target, transcript)

        if sim > best_score:
            best_score = sim
            best_transcript = transcript

    return best_transcript, best_score

In [53]:
aligned_rows = []

for _, row in df_targets.iterrows():
    target = row["target"]

    best_transcript, sim = find_best_match(target, df_transcript)

    aligned_rows.append({
        "target": target,
        "transcript": best_transcript if best_transcript else "",
        "similarity": sim
    })

df_aligned = pd.DataFrame(aligned_rows)

In [54]:
df_aligned["score"] = df_aligned.apply(
    lambda row: compute_score(row["target"], row["transcript"], row["similarity"]),
    axis=1
)

In [55]:
df_aligned.head()

,target,transcript,similarity,score
0,Quiero cortarme el pelo,Me gustan las películas que acaban bien,0.231692,1
1,El libro está en la mesa,Le pedí a un amigo que me ayudara con la tarea,0.442764,2
2,El carro lo tiene Pedro,El chico con el que yo salgo es español,0.647633,3
3,El se ducha cada mañana,"Antes de poder salir, él tiene que limpiar su ...",0.479716,2
4,¿Qué dice usted que va a hacer hoy?,¿Qué dice usted que va a hacer hoy?,1.000000,5


In [58]:
df_aligned.to_csv("../results/scored_output.csv", index=False, encoding="utf-8-sig")